# A1 — CC Narrative vs. Table 3 Reconciliation

**Reviewer concern addressed:** Section A1 of the revision checklist — *"Table 3 shows agents have higher Spearman ρ with CC (0.320 vs 0.286), yet the text repeatedly claims developers 'aggressively scale' with CC. The quantile slopes (0.624 vs 0.671) are near-identical. Revise the narrative or run a significance test on the slope difference."*

This notebook:
1. Re-runs Spearman correlations between `doc_tokens` and `CC`/`SLOC` to confirm Table 3 values.
2. Tests whether the between-group Spearman ρ difference is statistically significant (Fisher r-to-z).
3. Computes 95% confidence intervals for quantile regression slopes (CC) for both groups.
4. Runs a targeted Mann–Whitney test for the Complex CC bin (the strongest evidence for the narrative).
5. Produces a **summary narrative cell** with suggested revised paper text.
6. Saves outputs to `revision_outputs/cc_reconciliation_*.csv`.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, spearmanr, norm
import statsmodels.api as sm

sns.set_theme(style='whitegrid')
PALETTE = {'Agent': '#A7C7E7', 'Developer': '#BDE5B8'}

DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'dataset', 'data', 'updated_dataset_metrics.csv')
OUT_DIR   = os.path.join(os.getcwd(), 'revision_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
for col in ['doc_lines', 'doc_entropy', 'doc_code_overlap', 'doc_redundancy',
            'cyclomatic_complexity', 'sloc', 'semgrep_findings_count']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

df = df_raw.dropna(subset=['doc_entropy', 'doc_code_overlap', 'doc_redundancy']).copy()
df = df[df['doc_lines'] > 0].copy()

def tokenize(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'[A-Za-z_][A-Za-z0-9_]*', text.lower())

df['doc_tokens'] = df['doc_text'].apply(lambda x: len(tokenize(x)))
df['group_label'] = df['group'].map({'agent': 'Agent', 'human': 'Developer'})

df_a = df[df['group'] == 'agent'].copy()
df_h = df[df['group'] == 'human'].copy()

print(f"Agent: {len(df_a)} | Developer: {len(df_h)}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIRM TABLE 3 SPEARMAN CORRELATIONS
# ─────────────────────────────────────────────────────────────────────────────
print('=== Spearman ρ: doc_tokens ~ CC and doc_tokens ~ SLOC ===')

corr_rows = []

for predictor, pred_label in [('cyclomatic_complexity', 'CC'), ('sloc', 'SLOC')]:
    for grp_df, grp_name in [(df_a, 'Agent'), (df_h, 'Developer')]:
        mask = grp_df['doc_tokens'].notna() & grp_df[predictor].notna()
        x = grp_df.loc[mask, predictor]
        y = grp_df.loc[mask, 'doc_tokens']
        n = len(x)
        rho, p_val = spearmanr(x, y)
        corr_rows.append({
            'predictor': pred_label,
            'group': grp_name,
            'n': n,
            'spearman_rho': rho,
            'p_value': p_val,
        })
        print(f"  [{grp_name:9s}] doc_tokens ~ {pred_label}: ρ = {rho:.4f}  p = {p_val:.3e}  n = {n}")

corr_df = pd.DataFrame(corr_rows)

print()
print("Table 3 reference values: Agent CC ρ = 0.320, Developer CC ρ = 0.286")
print("(If the values above closely match, Table 3 is confirmed correct.)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. FISHER r-to-z TEST: Is the Spearman ρ difference statistically significant?
#
# Fisher's z-transformation converts Spearman ρ to a normally distributed z
# so we can test H₀: ρ_agent = ρ_developer.
# ─────────────────────────────────────────────────────────────────────────────

def fisher_rz_test(r1, n1, r2, n2):
    """
    Two-sided Fisher r-to-z test for H₀: ρ₁ = ρ₂.
    Returns (z_score, p_value).
    Clip correlations to avoid atanh(±1) = ±inf.
    """
    r1 = np.clip(r1, -0.9999, 0.9999)
    r2 = np.clip(r2, -0.9999, 0.9999)
    z1 = np.arctanh(r1)
    z2 = np.arctanh(r2)
    se = np.sqrt(1.0 / (n1 - 3) + 1.0 / (n2 - 3))
    z  = (z1 - z2) / se
    p  = 2 * (1 - norm.cdf(abs(z)))
    return z, p

print('=== Fisher r-to-z Test: Is ρ_agent ≠ ρ_developer? ===')

fisher_rows = []
for pred_label in ['CC', 'SLOC']:
    ag_row = corr_df[(corr_df['predictor'] == pred_label) & (corr_df['group'] == 'Agent')].iloc[0]
    hu_row = corr_df[(corr_df['predictor'] == pred_label) & (corr_df['group'] == 'Developer')].iloc[0]

    z, p = fisher_rz_test(ag_row['spearman_rho'], ag_row['n'],
                          hu_row['spearman_rho'], hu_row['n'])

    sig = 'significant' if p < 0.05 else 'NOT significant'
    print(f"  {pred_label}: z = {z:.4f}, p = {p:.4f}  → {sig}")
    print(f"    (ρ_agent = {ag_row['spearman_rho']:.4f}, ρ_developer = {hu_row['spearman_rho']:.4f}, "
          f"diff = {ag_row['spearman_rho'] - hu_row['spearman_rho']:+.4f})")

    fisher_rows.append({
        'predictor': pred_label,
        'rho_agent': ag_row['spearman_rho'],
        'rho_developer': hu_row['spearman_rho'],
        'rho_diff': ag_row['spearman_rho'] - hu_row['spearman_rho'],
        'fisher_z': z,
        'fisher_p': p,
        'significant': p < 0.05,
    })

fisher_df = pd.DataFrame(fisher_rows)
fisher_df.to_csv(os.path.join(OUT_DIR, 'cc_reconciliation_fisher.csv'), index=False)
print('Saved cc_reconciliation_fisher.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. QUANTILE REGRESSION SLOPES WITH 95% CIs
#    Replicates the analysis behind the paper's 0.624 / 0.671 slopes.
#    Bootstraps 95% CIs to test whether slopes overlap.
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== Quantile Regression Slopes (Median, q=0.5) with Bootstrap CIs ===')

N_BOOT = 1000
np.random.seed(42)

def qr_slope(sub_df, predictor='cyclomatic_complexity', outcome='doc_tokens', q=0.5):
    """Fit quantile regression and return median slope."""
    valid = sub_df[[predictor, outcome]].dropna()
    X = sm.add_constant(valid[predictor])
    model = sm.QuantReg(valid[outcome], X)
    result = model.fit(q=q)
    return result.params[predictor]

def bootstrap_qr_ci(sub_df, predictor='cyclomatic_complexity', outcome='doc_tokens', q=0.5, n_boot=N_BOOT):
    """Bootstrap 95% CI for quantile regression slope."""
    slopes = []
    valid = sub_df[[predictor, outcome]].dropna()
    for _ in range(n_boot):
        sample = valid.sample(frac=1, replace=True)
        X = sm.add_constant(sample[predictor])
        try:
            model = sm.QuantReg(sample[outcome], X)
            result = model.fit(q=q)
            slopes.append(result.params[predictor])
        except Exception:
            pass
    if not slopes:
        return np.nan, np.nan
    return np.percentile(slopes, 2.5), np.percentile(slopes, 97.5)

qr_rows = []
for grp_df, grp_name in [(df_a, 'Agent'), (df_h, 'Developer')]:
    slope = qr_slope(grp_df)
    ci_lo, ci_hi = bootstrap_qr_ci(grp_df)
    overlaps = None  # computed after both are available
    print(f"  [{grp_name:9s}] QR slope (CC): {slope:.4f}  95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")
    qr_rows.append({'group': grp_name, 'qr_slope': slope, 'ci_lo': ci_lo, 'ci_hi': ci_hi})

qr_df = pd.DataFrame(qr_rows)

# Test CI overlap
ag_qr  = qr_df[qr_df['group'] == 'Agent'].iloc[0]
hu_qr  = qr_df[qr_df['group'] == 'Developer'].iloc[0]
ci_overlap = not (ag_qr['ci_hi'] < hu_qr['ci_lo'] or hu_qr['ci_hi'] < ag_qr['ci_lo'])

print(f"\n  CI overlap: {ci_overlap}")
if ci_overlap:
    print("  *** CIs OVERLAP — the difference in QR slopes is NOT statistically supported.")
    print("      The claim that developers 'aggressively scale' more than agents with CC")
    print("      is NOT supported by the quantile regression evidence alone.")
else:
    print("  CIs do NOT overlap — the slope difference is statistically distinguishable.")

qr_df['ci_overlap_with_other_group'] = ci_overlap
qr_df.to_csv(os.path.join(OUT_DIR, 'cc_reconciliation_qr_slopes.csv'), index=False)
print('Saved cc_reconciliation_qr_slopes.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. COMPLEX CC BIN — Mann-Whitney with effect size
#    This is the most direct evidence for the narrative claim.
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== Complex CC Bin (CC ≥ 11): Mann-Whitney test ===')

CC_BINS   = [0, 2, 10, float('inf')]
CC_LABELS = ['Simple (1-2)', 'Moderate (3-10)', 'Complex (11+)']

df['cc_bin'] = pd.cut(df['cyclomatic_complexity'], bins=CC_BINS, labels=CC_LABELS)

bin_rows = []
for b in CC_LABELS:
    sub = df[df['cc_bin'] == b]
    ag_tok = sub[sub['group'] == 'agent']['doc_tokens']
    hu_tok = sub[sub['group'] == 'human']['doc_tokens']

    n_ag, n_hu = len(ag_tok), len(hu_tok)
    if n_ag < 2 or n_hu < 2:
        print(f"  {b}: too few observations (agent={n_ag}, human={n_hu})")
        continue

    stat, p = mannwhitneyu(ag_tok, hu_tok, alternative='two-sided')
    r = 1 - (2 * stat) / (n_ag * n_hu)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

    print(f"  {b}:")
    print(f"    n_agent={n_ag}, n_developer={n_hu}")
    print(f"    agent median={ag_tok.median():.1f}, developer median={hu_tok.median():.1f}")
    print(f"    U={stat:.0f}, p={p:.3e} {sig}, rank-biserial r={r:.4f}")

    bin_rows.append({
        'cc_bin': b, 'n_agent': n_ag, 'n_developer': n_hu,
        'agent_median': ag_tok.median(), 'developer_median': hu_tok.median(),
        'U_stat': stat, 'p_value': p, 'rank_biserial_r': r, 'significance': sig
    })

bin_df = pd.DataFrame(bin_rows)
bin_df.to_csv(os.path.join(OUT_DIR, 'cc_reconciliation_bins.csv'), index=False)
print('Saved cc_reconciliation_bins.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. VISUALISATION — scatter with QR lines for agents vs developers
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

for ax, grp_df, grp_name, color in [
    (axes[0], df_a, 'Agent', '#A7C7E7'),
    (axes[1], df_h, 'Developer', '#BDE5B8')
]:
    valid = grp_df[['cyclomatic_complexity', 'doc_tokens']].dropna()
    # Cap for readability
    valid = valid[valid['cyclomatic_complexity'] <= valid['cyclomatic_complexity'].quantile(0.99)]
    valid = valid[valid['doc_tokens']            <= valid['doc_tokens'].quantile(0.99)]

    ax.scatter(valid['cyclomatic_complexity'], valid['doc_tokens'],
               alpha=0.2, s=8, color=color, label='data')

    # QR line
    cc_range = np.linspace(valid['cyclomatic_complexity'].min(), valid['cyclomatic_complexity'].max(), 100)
    X_fit = sm.add_constant(valid['cyclomatic_complexity'])
    qr_fit = sm.QuantReg(valid['doc_tokens'], X_fit).fit(q=0.5)
    pred = qr_fit.params[0] + qr_fit.params[1] * cc_range
    ax.plot(cc_range, pred, color='red', linewidth=2,
            label=f'QR slope={qr_fit.params[1]:.3f}')

    ax.set_title(f'{grp_name}: doc_tokens vs CC', fontsize=13, fontweight='bold')
    ax.set_xlabel('Cyclomatic Complexity', fontsize=12)
    ax.set_ylabel('Documentation Tokens', fontsize=12)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'cc_reconciliation_scatter.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved cc_reconciliation_scatter.png')

## Synthesis: What the data supports and what it does not

Fill this cell after running the notebook with the actual numbers.

---

### What the data SUPPORTS

1. **Agents correlate with CC at least as strongly as developers.** Table 3 shows ρ_agent = 0.320 > ρ_developer = 0.286 for CC. The Fisher r-to-z test above indicates whether this difference is statistically significant. If not significant (p > 0.05), both groups scale similarly with CC.

2. **In the Complex CC bin (CC ≥ 11), agents document more tokens.** The Mann–Whitney test for this bin provides the most direct evidence that agents produce disproportionately longer documentation for highly complex functions. Report the effect size (rank-biserial r) and sample size here.

3. **Both groups increase documentation tokens with CC complexity.** The quantile regression slopes are both positive and in the same direction.

---

### What the data does NOT support

1. **The claim that developers 'aggressively scale' documentation with CC more than agents.** The slopes (≈0.624 vs ≈0.671) differ by only ~0.05, and if their 95% CIs overlap, this difference is not statistically distinguishable. This specific phrasing should be revised.

2. **The direction of the Spearman ρ difference.** Agents have *higher* ρ with CC (0.320 vs 0.286). If the narrative claims developers respond more to CC, this directly contradicts Table 3.

---

### Suggested revised narrative for Discussion 7.1 / Section 4.2

> *Both agents and developers increase documentation length with cyclomatic complexity (Spearman ρ = 0.320 and 0.286, respectively; Table 3). Contrary to our initial interpretation, the correlation between documentation length and CC is marginally stronger for agents than for developers, and the between-group difference is [not / marginally] statistically significant (Fisher r-to-z: z = [Z], p = [P]). Quantile regression slopes for the median (0.624 for agents, 0.671 for developers) are near-identical with overlapping confidence intervals ([CI_agent] vs [CI_dev]), providing no strong evidence of differential scaling in the central tendency. The most direct evidence for differential CC sensitivity comes from the Complex CC bin (CC ≥ 11), where agents produce [X] tokens (median) versus developers' [Y] tokens (U = [U], p = [p], r = [r], [effect size label]).*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# AUTO-POPULATE THE NARRATIVE TEMPLATE WITH ACTUAL NUMBERS
# ─────────────────────────────────────────────────────────────────────────────
cc_fisher = fisher_df[fisher_df['predictor'] == 'CC'].iloc[0]
sloc_fisher = fisher_df[fisher_df['predictor'] == 'SLOC'].iloc[0]
ag_qr_row = qr_df[qr_df['group'] == 'Agent'].iloc[0]
hu_qr_row = qr_df[qr_df['group'] == 'Developer'].iloc[0]

# Complex CC bin
complex_row = bin_df[bin_df['cc_bin'] == 'Complex (11+)'].iloc[0] if 'Complex (11+)' in bin_df['cc_bin'].values else None

print('=== SUGGESTED REVISED NARRATIVE (auto-populated) ===')
print()
print(f"Spearman ρ: agent CC = {cc_fisher['rho_agent']:.3f}, developer CC = {cc_fisher['rho_developer']:.3f}")
print(f"Fisher r-to-z (CC): z = {cc_fisher['fisher_z']:.3f}, p = {cc_fisher['fisher_p']:.4f} "
      f"({'significant' if cc_fisher['significant'] else 'NOT significant'})")
print()
print(f"QR slope: agent = {ag_qr_row['qr_slope']:.3f} [95%CI: {ag_qr_row['ci_lo']:.3f}–{ag_qr_row['ci_hi']:.3f}]")
print(f"QR slope: developer = {hu_qr_row['qr_slope']:.3f} [95%CI: {hu_qr_row['ci_lo']:.3f}–{hu_qr_row['ci_hi']:.3f}]")
print(f"CI overlap: {ci_overlap}")
print()

if complex_row is not None:
    print(f"Complex CC bin:")
    print(f"  Agent median tokens:     {complex_row['agent_median']:.0f}")
    print(f"  Developer median tokens: {complex_row['developer_median']:.0f}")
    print(f"  U = {complex_row['U_stat']:.0f}, p = {complex_row['p_value']:.3e}, "
          f"r = {complex_row['rank_biserial_r']:.4f} ({complex_row['significance']})")

print()
print('--- WHAT TO CHANGE IN THE PAPER ---')
if not cc_fisher['significant']:
    print("1. Remove or qualify any claim that developers 'aggressively scale' MORE than agents.")
    print("   The Fisher test shows the ρ difference is not statistically significant.")

if ci_overlap:
    print("2. The QR slope difference (0.624 vs 0.671) is not statistically supported — CIs overlap.")
    print("   Rephrase to: 'both groups show similar median scaling with CC.'")

if cc_fisher['rho_agent'] > cc_fisher['rho_developer']:
    print("3. Agents have HIGHER ρ with CC than developers in Table 3.")
    print("   Any text saying developers respond *more* to CC must be corrected.")

if complex_row is not None and complex_row['significance'] not in ['ns']:
    print(f"4. The Complex CC bin IS significant (p={complex_row['p_value']:.3e}). This is the")
    print("   strongest evidence for differential documentation depth in high-complexity code.")
    print("   Focus the narrative on this bin comparison, not the global slopes.")

## Revision note

- **What this adds:** Statistical resolution to the Table 3 / Discussion conflict. The Fisher r-to-z test definitively answers whether agents and developers respond *differently* to CC in terms of documentation length. The QR slope CIs answer whether the near-identical slopes (0.624 vs 0.671) can support a differential-scaling claim.
- **Where to cite in paper:** Section 4.2.2 and Discussion 7.1. Replace narrative claims about 'aggressive scaling' with the more nuanced finding from the Complex CC bin (the most defensible evidence).
- **What to watch for:** If the Complex CC bin result is also not significant after BH correction (see `rq_fix_statistics.ipynb`), the differential-scaling claim loses all statistical support and must be reframed as a descriptive/exploratory observation.